# SAFEWORK AI: latih lanjutan model APD (v2)

Notebook ini dijalankan sekali **Run All** di Colab atau Kaggle (GPU T4). Ia melanjutkan model `ppe-v1` supaya kelas lama tetap terjaga, lalu menambah dua kelas baru: `mask` dan `no_mask`.

1. Unduh dua dataset publik tanpa API key: Construction-PPE dan PPEs Dataset.
2. Gabungkan menjadi 14 kelas. Dari PPEs hanya split `valid` dan `test` yang dipakai, karena split `train`-nya tidak berisi mask, helmet, maupun shoes (data PPEs dibagi per sumber). Gambar kedua split digabung lalu dibagi acak.
3. Lengkapi label yang tidak ada di PPEs (Person, vest, no_vest, forklift) dengan model, supaya model tidak belajar bahwa orang dan rompi adalah latar belakang.
4. Latih lanjutan dari `ppe-v1`, lalu bandingkan dengan `ppe-v1` pada 200 gambar validasi Construction-PPE yang sama.

| Sumber | Isi | Lisensi |
|---|---|---|
| Ultralytics Construction-PPE (rilis 3 Nov 2026) | 1.000 gambar, 12 kelas termasuk `no_*` | CC BY 4.0 |
| PPEs Dataset, Roboflow Universe (salinan Hugging Face `keremberke/protective-equipment-detection`) | 11.978 gambar: glove, goggles, helmet, mask, shoes dan pasangan `no_*` | CC BY 4.0 |

Penjaga mutu: sel 6 dan 8 menghentikan proses bila gambar PPEs terlalu sedikit atau `mask` dan `no_mask` hampir kosong di data latih, supaya jam GPU tidak terbuang. Keluaran sel 6 memuat nama kategori asli dataset beserta pemetaannya.

Hasil latihan masuk folder `ppe-v2-ft1` (akhiran dari `RUN_TAG` di sel 2), jadi run lama tidak ikut terbaca atau membuat latihan dilewati.

Perkiraan waktu: sekitar 45 sampai 60 menit di T4 (kasar, bergantung antrean CPU).

**Colab (browser atau VS Code):** pilih runtime GPU T4. Di VS Code: Select Kernel, Colab, New Colab Server, T4, lalu jalankan `Colab: Mount Google Drive to Server...` sebelum Run All.
**Kaggle:** buat notebook baru, File > Import Notebook, unggah file ini. Di panel Settings set Accelerator ke GPU T4 dan Internet ke On (keduanya butuh akun terverifikasi telepon). Hasil ada di `/kaggle/working/out/`.

Wajib: cantumkan kedua sumber di README atau dokumen lomba (atribusi CC BY 4.0).

In [1]:
# 1) Persiapan: paket, GPU, penyimpanan (Colab: Google Drive opsional; Kaggle: /kaggle/working)
import os, sys, re, json, shutil, random, zipfile, subprocess, collections
from pathlib import Path

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError("Gagal memasang ultralytics. Di Kaggle, aktifkan Internet di panel Settings (butuh verifikasi telepon).") from e

import torch
assert torch.cuda.is_available(), "GPU tidak aktif. Colab: pilih runtime GPU T4. Kaggle: Settings > Accelerator > GPU T4."
print("GPU:", torch.cuda.get_device_name(0))

import ultralytics
print("ultralytics", ultralytics.__version__)

# Matikan logger pihak ketiga supaya latihan tidak berhenti untuk meminta login
os.environ["WANDB_MODE"] = "disabled"
for key in ("wandb", "clearml", "comet", "mlflow", "neptune", "tensorboard"):
    try:
        ultralytics.settings.update({key: False})
    except Exception:
        pass  # kunci tidak ada di versi ini: abaikan

IS_KAGGLE = Path("/kaggle/working").exists()
DRIVE_DIR = None
if IS_KAGGLE:
    WORK = Path("/kaggle/temp/work") if Path("/kaggle/temp").exists() else Path("/kaggle/working/work")
    RUNS = Path("/kaggle/working/runs")
    OUT = Path("/kaggle/working/out")
else:
    USE_DRIVE = True  # simpan checkpoint dan hasil ke Google Drive (disarankan; sesi Colab bisa putus)
    if USE_DRIVE:
        try:
            if not Path("/content/drive/MyDrive").exists():  # belum di-mount (di VS Code: jalankan dulu "Colab: Mount Google Drive to Server...")
                from google.colab import drive
                drive.mount("/content/drive")
            DRIVE_DIR = Path("/content/drive/MyDrive/safework_ppe_v2")
            DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            DRIVE_DIR = None
            print("Drive tidak terhubung, hasil hanya di /content:", e)
    WORK = Path("/content/work")
    RUNS = (DRIVE_DIR / "runs") if DRIVE_DIR else Path("/content/runs")
    OUT = Path("/content/out")
for d in (WORK, RUNS, OUT):
    d.mkdir(parents=True, exist_ok=True)
print("Platform:", "Kaggle" if IS_KAGGLE else "Colab/lokal", "| hasil latihan:", RUNS)

GPU: Tesla T4
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.216
Mounted at /content/drive
Platform: Colab/lokal | hasil latihan: /content/drive/MyDrive/safework_ppe_v2/runs


In [2]:
# 2) Pengaturan: ubah di sini saja
INIT_WEIGHTS = None          # None = lanjut dari ppe-v1 (BASELINE_WEIGHTS): 12 kelas lama terjaga, mask/no_mask ditambah. Isi "yolo11n.pt" untuk mulai dari COCO (lebih lama)
RUN_TAG = "ft1"              # akhiran nama folder hasil. Tag baru = latihan baru; tag yang sama = lanjut dari checkpoint, atau dilewati bila sudah selesai
EPOCHS = 30
MAX_HOURS = None             # mis. 0.75 = latih tepat sekitar 45 menit (jumlah epoch menyesuaikan, jadwal belajar ikut). None = pakai EPOCHS
PATIENCE = 10
IMGSZ = 640
BATCH = 32
CACHE = "disk"               # cache gambar terubah-ukuran agar loader (2-4 CPU) tidak jadi hambatan. False = mati
WORKERS = max(2, min(4, os.cpu_count() or 2))
SEED = 0
EXTRA_TRAIN_ARGS = {}        # argumen latih tambahan, mis. {"cls_pw": 0.25} bila kelas langka (no_*) masih lemah setelah latihan pertama

CPPE_OVERSAMPLE = 2          # Construction-PPE (label paling bersih) diulang agar tidak tenggelam oleh PPEs
PPES_SPLITS = ["valid", "test"]  # split 'train' PPEs tidak berisi mask/helmet/shoes; tambahkan "train" hanya bila mau lebih banyak gloves/goggles
PPES_HOLDOUT = 0.15          # porsi gambar PPEs yang disisihkan untuk validasi dan uji (dibagi dua)
PPES_MAX_TRAIN = 3000        # batas gambar PPEs untuk latih; gambar berisi kelas wajib (mask/no_mask) selalu ikut. None = semua
PSEUDO_CONF = 0.5            # ambang keyakinan label otomatis (Person, vest, no_vest, forklift)

BASELINE_WEIGHTS = None      # None = ambil best.pt dari repo GitHub. Atau isi path best.pt yang kamu unggah
REPO_URL = "https://github.com/wildnrfm/iot-safework.git"
CPPE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
HF_BASE = "https://huggingface.co/datasets/keremberke/protective-equipment-detection/resolve/main/data/"

# Penjaga mutu data: proses berhenti lebih awal (sebelum menghabiskan jam GPU) bila datanya tidak masuk akal
REQUIRED_CLASSES = ["mask", "no_mask"]   # kelas yang wajib punya contoh latih
MIN_TRAIN_INSTANCES = 100                # minimal contoh latih per kelas wajib
MIN_PPES_POOL_IMAGES = 3000              # valid + test seharusnya 5.505 gambar
ALLOW_MISSING = False                    # True = tetap latih walau penjaga di atas gagal (tidak disarankan)

# 14 kelas akhir. 12 pertama = urutan resmi Construction-PPE Nov 2026 (sama dengan ppe-v1), lalu mask dan no_mask.
CLASSES = ["helmet", "gloves", "vest", "boots", "goggles", "Person", "no_helmet", "no_goggle",
           "no_gloves", "no_boots", "forklift", "no_vest", "mask", "no_mask"]
NAME2ID = {n: i for i, n in enumerate(CLASSES)}
N_CPPE = 12

# PPEs -> kelas akhir (nama dinormalkan: huruf kecil, '-' jadi '_'). Kelas lain (suit, no_suit) dibuang.
# Catatan: 'shoes' di PPEs = sepatu umum, sedangkan 'boots' di Construction-PPE = sepatu kerja pelindung.
PPES_MAP = {"glove": "gloves", "goggles": "goggles", "helmet": "helmet", "mask": "mask",
            "no_glove": "no_gloves", "no_goggles": "no_goggle", "no_helmet": "no_helmet",
            "no_mask": "no_mask", "no_shoes": "no_boots", "shoes": "boots"}

In [3]:
# 3) Fungsi bantu (hanya definisi; belum ada yang dijalankan)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def run_name():
    return f"ppe-v2-{RUN_TAG}"


def norm(name):
    return re.sub(r"[\s\-]+", "_", str(name).strip().lower())


def safe_name(name):
    return re.sub(r"[^A-Za-z0-9_.\-]", "_", str(name))


def download(url, dest):
    import requests
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print("sudah ada:", dest.name)
        return dest
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    tmp.rename(dest)
    print(f"selesai: {dest.name} ({dest.stat().st_size / 1e6:.0f} MB)")
    return dest


def unzip(zpath, out_dir):
    out_dir = Path(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(out_dir)
    return out_dir


def list_images(d):
    d = Path(d)
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT) if d.is_dir() else []


def write_label(path, lines):
    Path(path).write_text("".join(l + "\n" for l in lines))


PPES_FAMILIES = {"mask": ("mask", "no_mask"), "glove": ("gloves", "no_gloves"), "goggle": ("goggles", "no_goggle"),
                 "helmet": ("helmet", "no_helmet"), "hardhat": ("helmet", "no_helmet"),
                 "shoe": ("boots", "no_boots"), "boot": ("boots", "no_boots")}


def map_ppes_name(name):
    """Nama kategori PPEs -> kelas akhir. Cocok persis (PPES_MAP) dulu, lalu tebak dari kata kunci; None = dibuang."""
    n = norm(name)
    if n in PPES_MAP:
        return PPES_MAP[n]
    if "suit" in n or "incorrect" in n:
        return None
    neg = bool(re.search(r"(^|_)(no|non|without)(_|$)|(^|_)no(mask|glove|goggle|helmet|hardhat|shoe|boot)", n))
    for key, (pos, negname) in PPES_FAMILIES.items():
        if key in n:
            return negname if neg else pos
    return None


def coco_to_yolo_lines(img_w, img_h, anns, catid2name, name_map):
    """Anotasi COCO (x, y, w, h piksel) -> baris YOLO (kelas xc yc w h, 0-1). Kotak dipotong ke batas gambar."""
    lines = []
    for a in anns:
        cname = catid2name.get(a.get("category_id"), "")
        tgt = name_map(cname) if callable(name_map) else name_map.get(norm(cname))
        if tgt is None:
            continue
        x, y, w, h = a["bbox"]
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(float(img_w), x + w), min(float(img_h), y + h)
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue
        lines.append(f"{NAME2ID[tgt]} {(x1 + x2) / 2 / img_w:.6f} {(y1 + y2) / 2 / img_h:.6f} {bw / img_w:.6f} {bh / img_h:.6f}")
    return lines


def convert_ppes(src_dir, img_out, lbl_out, prefix, keep=None):
    """PPEs (COCO) -> YOLO. keep: himpunan nama file untuk subset (None = semua). Mencetak nama kategori asli dan pemetaannya."""
    cands = sorted(Path(src_dir).rglob("_annotations.coco.json"))
    assert cands, f"_annotations.coco.json tidak ditemukan di {src_dir}"
    base = cands[0].parent
    data = json.loads(cands[0].read_text())
    catid2name = {c["id"]: c["name"] for c in data["categories"]}
    raw = collections.Counter(catid2name.get(a.get("category_id"), f"id{a.get('category_id')}") for a in data["annotations"])
    targets = {n: map_ppes_name(n) for n in set(catid2name.values()) | set(raw)}
    per_img = collections.defaultdict(list)
    for a in data["annotations"]:
        per_img[a["image_id"]].append(a)
    img_out, lbl_out = Path(img_out), Path(lbl_out)
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    stems, boxes, wanted, missing = [], 0, 0, 0
    for im in data["images"]:
        if keep is not None and im["file_name"] not in keep:
            continue
        wanted += 1
        src = base / im["file_name"]
        if not src.exists():
            missing += 1
            continue
        lines = coco_to_yolo_lines(im["width"], im["height"], per_img[im["id"]], catid2name, map_ppes_name)
        name = safe_name(f"{prefix}_{im['file_name']}")
        shutil.copy2(src, img_out / name)
        write_label(lbl_out / (Path(name).stem + ".txt"), lines)
        stems.append(Path(name).stem)
        boxes += len(lines)
    print(f"  {prefix}: {len(stems)} gambar, {boxes} kotak")
    print(f"    anotasi asli di split ini (nama: jumlah): {dict(raw.most_common())}")
    print(f"    dipetakan: { {n: t for n, t in sorted(targets.items()) if t} } | dibuang: {sorted(n for n, t in targets.items() if not t)}")
    if wanted and missing / wanted > 0.05:
        raise RuntimeError(f"{missing} dari {wanted} gambar di {prefix} tidak ditemukan di samping _annotations.coco.json; "
                           "cek struktur arsip yang diunduh.")
    if stems and boxes == 0:
        raise RuntimeError("Tidak ada kotak yang cocok dengan pemetaan kelas; cek nama kategori pada baris 'anotasi asli' di atas.")
    return stems


def has_class(label_path, ids):
    for ln in Path(label_path).read_text().splitlines():
        if ln.strip() and int(float(ln.split()[0])) in ids:
            return True
    return False


def split_pool(lbl_dir, holdout, max_train, seed, must_ids):
    """Bagi kolam gambar secara acak. Porsi holdout disisihkan dulu (dibagi dua: val dan test); sisanya calon latih.
    Gambar yang berisi kelas wajib (must_ids) selalu ikut latih; sisanya diisi acak sampai max_train (None = semua)."""
    lbl_dir = Path(lbl_dir)
    stems = sorted(p.stem for p in lbl_dir.glob("*.txt"))
    random.Random(seed).shuffle(stems)
    n_hold = int(len(stems) * holdout)
    val, test = stems[: n_hold // 2], stems[n_hold // 2: n_hold]
    rest = stems[n_hold:]
    must = [s for s in rest if has_class(lbl_dir / f"{s}.txt", must_ids)]
    must_set = set(must)
    others = [s for s in rest if s not in must_set]
    room = len(others) if max_train is None else max(0, max_train - len(must))
    return {"train": must + others[:room], "val": val, "test": test}


def move_stems(stems, src_img, src_lbl, dst_img, dst_lbl):
    src_img, src_lbl, dst_img, dst_lbl = Path(src_img), Path(src_lbl), Path(dst_img), Path(dst_lbl)
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    imgs = {p.stem: p for p in list_images(src_img)}
    for s in stems:
        shutil.move(str(imgs[s]), str(dst_img / imgs[s].name))
        shutil.move(str(src_lbl / f"{s}.txt"), str(dst_lbl / f"{s}.txt"))


def find_yolo_root(base):
    for d in sorted(Path(base).rglob("train")):
        if d.parent.name == "images" and (d.parent.parent / "labels" / "train").is_dir():
            return d.parent.parent
    raise FileNotFoundError(f"struktur images/train + labels/train tidak ditemukan di {base}")


def copy_cppe(root, split, img_out, lbl_out, repeat=1):
    """Salin Construction-PPE (ID kelas sudah sama dengan CLASSES[:12]); repeat = jumlah salinan."""
    root = Path(root)
    img_out, lbl_out = Path(img_out), Path(lbl_out)
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in list_images(root / "images" / split):
        lab = root / "labels" / split / (p.stem + ".txt")
        text = lab.read_text() if lab.exists() else ""
        for ln in text.splitlines():
            if ln.strip() and not 0 <= int(float(ln.split()[0])) < N_CPPE:
                raise ValueError(f"ID kelas di luar 0-{N_CPPE - 1} pada {lab.name}: bukan arsip Construction-PPE Nov 2026?")
        if text and not text.endswith("\n"):
            text += "\n"
        stem = safe_name(p.stem)
        for k in range(repeat):
            tag = "" if k == 0 else f"_x{k}"
            shutil.copy2(p, img_out / f"cppe_{stem}{tag}{p.suffix.lower()}")
            (lbl_out / f"cppe_{stem}{tag}.txt").write_text(text)
        n += 1
    return n


def complete_labels(model_path, img_dir, lbl_dir, name_map, conf, classes=None, batch=32):
    """Tambahkan deteksi model untuk kelas yang tidak dilabeli sumber. name_map: nama kelas model (dinormalkan) -> kelas akhir."""
    from ultralytics import YOLO
    model = YOLO(str(model_path))
    added = collections.Counter()
    for r in model.predict(source=str(img_dir), conf=conf, imgsz=IMGSZ, batch=batch, classes=classes,
                           stream=True, verbose=False, device=0):
        if r.boxes is None or len(r.boxes) == 0:
            continue
        lines = []
        for c, (x, y, w, h) in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist()):
            tgt = name_map.get(norm(r.names[int(c)]))
            if tgt is None:
                continue
            lines.append(f"{NAME2ID[tgt]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
            added[tgt] += 1
        if lines:
            with open(Path(lbl_dir) / (Path(r.path).stem + ".txt"), "a") as f:
                f.write("".join(l + "\n" for l in lines))
    return added


def count_instances(lbl_dir):
    c = collections.Counter()
    for f in Path(lbl_dir).glob("*.txt"):
        for ln in f.read_text().splitlines():
            if ln.strip():
                c[CLASSES[int(float(ln.split()[0]))]] += 1
    return c


def write_yaml(path, root, train, val, names=None):
    names = names or CLASSES
    fmt = lambda x: ("[" + ", ".join(x) + "]") if isinstance(x, (list, tuple)) else x
    lines = [f"path: {root}", f"train: {fmt(train)}", f"val: {fmt(val)}", "names:"]
    lines += [f"  {i}: {n}" for i, n in enumerate(names)]
    Path(path).write_text("\n".join(lines) + "\n")


def eval_per_class(weights, data_yaml):
    """Validasi satu model; kembalikan {kelas: {p, r, ap50, ap}}, mAP50, mAP50-95 (hanya kelas yang punya contoh)."""
    from ultralytics import YOLO
    model = YOLO(str(weights))
    m = model.val(data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=0.6, plots=False, verbose=False)
    b = m.box
    out = {}
    for i, c in enumerate(b.ap_class_index):
        try:
            p, r, ap50, ap = b.class_result(i)
        except AttributeError:  # cadangan bila metode ini berganti nama
            p, r, ap50, ap = b.p[i], b.r[i], b.ap50[i], b.ap[i]
        out[model.names[int(c)]] = {"p": float(p), "r": float(r), "ap50": float(ap50), "ap": float(ap)}
    return out, float(b.map50), float(b.map)

## Data

Sel 5 mengunduh sekitar 2 GB (Construction-PPE dari GitHub Ultralytics, PPEs dari Hugging Face) dan mengambil `best.pt` baseline dari repo GitHub kamu. Sel 6 sampai 8 menggabungkan, melengkapi label, dan mencetak jumlah objek per kelas.

In [4]:
# 5) Unduh data dan ambil model baseline
cppe_zip = download(CPPE_URL, WORK / "construction-ppe.zip")
cppe_raw = unzip(cppe_zip, WORK / "raw_cppe")

ppes_raw = {}
for split in PPES_SPLITS:
    z = download(HF_BASE + f"{split}.zip", WORK / f"ppes_{split}.zip")
    ppes_raw[split] = unzip(z, WORK / f"raw_ppes_{split}")

# Baseline ppe-v1 dari repo: titik awal latihan, pelengkap label vest/no_vest/forklift, dan pembanding
if not BASELINE_WEIGHTS:
    repo_dir = WORK / "repo"
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
    found = sorted(repo_dir.rglob("best.pt"), key=lambda p: ("ppe-v1" not in str(p), str(p)))
    BASELINE_WEIGHTS = str(found[0]) if found else None
assert BASELINE_WEIGHTS and Path(BASELINE_WEIGHTS).exists() and Path(BASELINE_WEIGHTS).stat().st_size > 1_000_000, \
    "best.pt baseline tidak ditemukan. Unggah ke notebook lalu isi BASELINE_WEIGHTS di sel 2."
print("baseline:", BASELINE_WEIGHTS)

selesai: construction-ppe.zip (648 MB)
selesai: ppes_valid.zip (1126 MB)
selesai: ppes_test.zip (245 MB)
baseline: /content/work/repo/runs/ppe-v1/weights/best.pt


In [5]:
# 6) Gabungkan ke satu dataset YOLO 14 kelas (dibangun ulang dari nol tiap dijalankan)
MERGED = WORK / "ppe_v2"
shutil.rmtree(MERGED, ignore_errors=True)
IMG, LBL = MERGED / "images", MERGED / "labels"

# 6a) Construction-PPE: label paling bersih, diulang CPPE_OVERSAMPLE kali di data latih
cppe_root = find_yolo_root(cppe_raw)
n_all = sum(len(list_images(cppe_root / "images" / s)) for s in ("train", "val", "test"))
if n_all == 1416 or (cppe_root / "images" / "test").is_dir():
    raise RuntimeError("Arsip Construction-PPE ini versi lama (1.416 gambar, 11 kelas). Hapus "
                       "construction-ppe.zip dan raw_cppe di folder kerja, lalu jalankan ulang dari sel 5.")
n_tr = copy_cppe(cppe_root, "train", IMG / "train_cppe", LBL / "train_cppe", repeat=CPPE_OVERSAMPLE)
n_va = copy_cppe(cppe_root, "val", IMG / "val_cppe", LBL / "val_cppe")
print(f"Construction-PPE: train {n_tr} (x{CPPE_OVERSAMPLE}), val {n_va}")
if (n_tr, n_va) != (800, 200):
    print("PERINGATAN: jumlah gambar bukan 800/200 seperti rilis 3 Nov 2026; pastikan versi dan urutan kelasnya benar.")

# 6b) PPEs: kelas tersebar menurut split (train: gloves/goggles; valid: campuran termasuk mask; test: helmet).
#     Karena itu split yang dipilih digabung jadi satu kolam, lalu dibagi acak per gambar.
n_pool = 0
for split in PPES_SPLITS:
    n_pool += len(convert_ppes(ppes_raw[split], IMG / "pool_ppes", LBL / "pool_ppes", f"ppes_{split}"))
print(f"Kolam PPEs: {n_pool} gambar dari split {PPES_SPLITS}")
if n_pool < MIN_PPES_POOL_IMAGES and not ALLOW_MISSING:
    raise RuntimeError(f"Hanya {n_pool} gambar PPEs yang terbaca (seharusnya sekitar 5.505 untuk valid + test). "
                       "Proses dihentikan agar tidak menghabiskan jam GPU; tempel keluaran sel 5 dan 6 ke chat.")
parts = split_pool(LBL / "pool_ppes", PPES_HOLDOUT, PPES_MAX_TRAIN, SEED, [NAME2ID[c] for c in REQUIRED_CLASSES])
for part, stems in parts.items():
    move_stems(stems, IMG / "pool_ppes", LBL / "pool_ppes", IMG / f"{part}_ppes", LBL / f"{part}_ppes")
shutil.rmtree(IMG / "pool_ppes", ignore_errors=True)   # sisa gambar yang tidak terpilih dibuang
shutil.rmtree(LBL / "pool_ppes", ignore_errors=True)
print("PPEs dibagi:", {k: len(v) for k, v in parts.items()})

Construction-PPE: train 800 (x2), val 200
  ppes_valid: 3570 gambar, 7677 kotak
    anotasi asli di split ini (nama: jumlah): {'no_glove': 1548, 'no_goggles': 1384, 'goggles': 1192, 'glove': 984, 'no_mask': 640, 'shoes': 639, 'no_shoes': 525, 'helmet': 283, 'mask': 253, 'no_helmet': 229}
    dipetakan: {'glove': 'gloves', 'goggles': 'goggles', 'helmet': 'helmet', 'mask': 'mask', 'no_glove': 'no_gloves', 'no_goggles': 'no_goggle', 'no_helmet': 'no_helmet', 'no_mask': 'no_mask', 'no_shoes': 'no_boots', 'shoes': 'boots'} | dibuang: []
  ppes_test: 1935 gambar, 2884 kotak
    anotasi asli di split ini (nama: jumlah): {'helmet': 1240, 'no_helmet': 1067, 'goggles': 300, 'shoes': 102, 'no_goggles': 77, 'no_shoes': 61, 'no_mask': 21, 'mask': 16}
    dipetakan: {'glove': 'gloves', 'goggles': 'goggles', 'helmet': 'helmet', 'mask': 'mask', 'no_glove': 'no_gloves', 'no_goggles': 'no_goggle', 'no_helmet': 'no_helmet', 'no_mask': 'no_mask', 'no_shoes': 'no_boots', 'shoes': 'boots'} | dibuang: []
Kol

In [6]:
# 7) Lengkapi label yang tidak ada di PPEs: Person (yolo11m COCO), vest/no_vest/forklift (baseline ppe-v1).
#    Tanpa ini model belajar bahwa orang dan rompi di gambar PPEs adalah latar belakang.
total = collections.Counter()
for split in ("train_ppes", "val_ppes", "test_ppes"):
    if not list_images(IMG / split):
        continue
    a = complete_labels("yolo11m.pt", IMG / split, LBL / split, {"person": "Person"}, PSEUDO_CONF, classes=[0])
    b = complete_labels(BASELINE_WEIGHTS, IMG / split, LBL / split,
                        {"vest": "vest", "no_vest": "no_vest", "forklift": "forklift"}, PSEUDO_CONF)
    print(split, dict(a + b))
    total.update(a + b)

train_ppes {'Person': 6275, 'no_vest': 3736, 'forklift': 30, 'vest': 211}
val_ppes {'Person': 899, 'no_vest': 546, 'vest': 20, 'forklift': 3}
test_ppes {'Person': 912, 'vest': 27, 'no_vest': 542, 'forklift': 4}


In [7]:
# 8) Berkas konfigurasi dan ringkasan jumlah objek per kelas
write_yaml(MERGED / "ppe_v2.yaml", MERGED, ["images/train_cppe", "images/train_ppes"], ["images/val_cppe", "images/val_ppes"])
write_yaml(MERGED / "eval_cppe.yaml", MERGED, "images/train_cppe", "images/val_cppe")
write_yaml(MERGED / "eval_ppes.yaml", MERGED, "images/train_cppe", ["images/val_ppes", "images/test_ppes"])
write_yaml(MERGED / "baseline_cppe.yaml", MERGED, "images/train_cppe", "images/val_cppe", names=CLASSES[:N_CPPE])

rows = {s: count_instances(LBL / s) for s in ("train_cppe", "train_ppes", "val_cppe", "val_ppes", "test_ppes")}
print(f"{'kelas':<11}" + "".join(f"{s:>12}" for s in rows))
for c in CLASSES:
    print(f"{c:<11}" + "".join(f"{rows[s][c]:>12}" for s in rows))
low = [c for c in CLASSES if rows["train_cppe"][c] + rows["train_ppes"][c] < 150]
if low:
    print("PERINGATAN: contoh latih sedikit (<150) untuk:", low)

# Penjaga: jangan lanjut ke latihan bila kelas wajib hampir kosong (latihan memakan jam GPU)
need = {c: rows["train_cppe"][c] + rows["train_ppes"][c] for c in REQUIRED_CLASSES}
print("contoh latih kelas wajib:", need)
too_few = [c for c, n in need.items() if n < MIN_TRAIN_INSTANCES]
if too_few and not ALLOW_MISSING:
    raise RuntimeError(f"Berhenti sebelum latihan: kelas {too_few} punya kurang dari {MIN_TRAIN_INSTANCES} contoh di data latih ({need}). "
                       "Lihat baris 'anotasi asli' dan 'dipetakan' di keluaran sel 6 (nama kategori sebenarnya ada di sana), "
                       "lalu tempel keluaran sel 6 dan 8 ke chat. Untuk tetap melatih, set ALLOW_MISSING = True di sel 2.")

kelas        train_cppe  train_ppes    val_cppe    val_ppes   test_ppes
helmet             2564         730         294          95         105
gloves             3096         487         381          89          76
vest               2952         211         340          20          27
boots              4006         391         481          71          46
goggles            1224         883         139         104         115
Person             3924        6275         481         899         912
no_helmet          1268         633         179          91         103
no_goggle           530         903          82         105         100
no_gloves          1678         867         203         113         122
no_boots           1138         307         178          48          39
forklift            550          30          73           3           4
no_vest             874        3736         130         546         542
mask                  0         236           0          15     

In [8]:
# 9) Latih lanjutan dari ppe-v1 (kepala klasifikasi untuk 12 kelas lama disalin otomatis oleh Ultralytics;
#    hanya mask dan no_mask yang baru). Bila ada last.pt dari sesi sebelumnya (tag yang sama), dilanjutkan.
from ultralytics import YOLO

init = INIT_WEIGHTS or BASELINE_WEIGHTS
name = run_name()
wdir = RUNS / name / "weights"
done = False
if (wdir / "last.pt").exists():
    print("Ada checkpoint, mencoba melanjutkan:", wdir / "last.pt")
    try:
        YOLO(str(wdir / "last.pt")).train(resume=True)
        done = True
    except Exception as e:
        print("Tidak bisa dilanjutkan (biasanya karena latihan sudah selesai):", e)
        done = (wdir / "best.pt").exists()
if not done:
    args = dict(
        data=str(MERGED / "ppe_v2.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
        workers=WORKERS, cache=CACHE, seed=SEED, device=0, close_mosaic=5, save_period=5,
        project=str(RUNS), name=name, exist_ok=True, plots=True,
    )
    if MAX_HOURS:
        args["time"] = MAX_HOURS
    args.update(EXTRA_TRAIN_ARGS)
    print("Mulai dari:", init)
    YOLO(str(init)).train(**args)

Mulai dari: /content/work/repo/runs/ppe-v1/weights/best.pt
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/ppe_v2/ppe_v2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/work/repo/runs/ppe-v1/weights/best.pt, momentum=0.93

In [9]:
# 10) Uji lolos atau tidak: bandingkan dengan baseline pada 200 gambar validasi Construction-PPE yang sama
def fmt(d, k):
    return f"{d[k]:.2f}" if d else "-"


base, base_map50, _ = eval_per_class(BASELINE_WEIGHTS, MERGED / "baseline_cppe.yaml")
print(f"baseline ppe-v1: mAP50 di Construction-PPE val = {base_map50:.3f}")

name = run_name()
best = RUNS / name / "weights" / "best.pt"
if not best.exists():
    print("best.pt belum ada untuk", name)
else:
    new_c, map_c, _ = eval_per_class(best, MERGED / "eval_cppe.yaml")
    new_p, map_p, _ = eval_per_class(best, MERGED / "eval_ppes.yaml")
    print(f"\n== {name}: mAP50 Construction-PPE val {map_c:.3f} (baseline {base_map50:.3f}) | PPEs holdout {map_p:.3f} ==")
    print(f"{'kelas':<11}{'base AP50':>10}{'v2 AP50':>9}{'v2 recall':>10}{'PPEs AP50':>11}{'recall':>8}")
    for c in CLASSES:
        b, n, t = base.get(c), new_c.get(c), new_p.get(c)
        print(f"{c:<11}{fmt(b, 'ap50'):>10}{fmt(n, 'ap50'):>9}{fmt(n, 'r'):>10}{fmt(t, 'ap50'):>11}{fmt(t, 'r'):>8}")
    turun = [c for c, b in base.items() if b["ap50"] >= 0.3 and c in new_c and new_c[c]["ap50"] < b["ap50"] - 0.05]
    if turun:
        print(f"\nPERHATIAN: AP50 turun lebih dari 0,05 pada {turun}. Naikkan CPPE_OVERSAMPLE atau kurangi EPOCHS lalu latih ulang dengan RUN_TAG baru.")
    else:
        print("\nLOLOS: tidak ada kelas lama yang turun lebih dari 0,05 AP50 dibanding baseline.")
    mask_ok = new_p.get("mask", {}).get("ap50", 0) >= 0.5 and new_p.get("no_mask", {}).get("ap50", 0) >= 0.4
    print("Patokan awal untuk memakai v2: LOLOS dan AP50 mask >= 0,5 dan no_mask >= 0,4 ->",
          "TERPENUHI" if (not turun and mask_ok) else "BELUM terpenuhi",
          "(contoh mask di holdout sedikit, jadi angkanya kasar; uji alat peraga tetap menentukan)")
    print("Catatan: pada kolom PPEs, baris Person, vest, no_vest, forklift memakai label otomatis; jangan dipakai sebagai ukuran.")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,599,724 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 141.9±57.1 MB/s, size: 538.3 KB)
val: Scanning /content/work/ppe_v2/labels/val_cppe... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 468.9it/s 0.4s
val: New cache created: /content/work/ppe_v2/labels/val_cppe.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.7it/s 4.2s
                   all        200       2961      0.871      0.709       0.77      0.544
Speed: 0.9ms preprocess, 3.7ms inference, 0.0ms loss, 1.7ms postprocess per image
baseline ppe-v1: mAP50 di Construction-PPE val = 0.770
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,600,210 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 

In [10]:
# 11) Simpan hasil
from ultralytics import YOLO

name = run_name()
best = RUNS / name / "weights" / "best.pt"
if best.exists():
    shutil.copy2(best, OUT / f"{name}.pt")
    if DRIVE_DIR:
        shutil.copy2(best, DRIVE_DIR / f"{name}.pt")
    print("tersimpan:", OUT / f"{name}.pt", "| kelas:", list(YOLO(str(best)).names.values()))
else:
    print("best.pt belum ada untuk", name)

if IS_KAGGLE:
    print("Kaggle: unduh file di panel Output (/kaggle/working/out) sebelum menutup sesi.")
else:
    try:
        from google.colab import files
        for f in sorted(OUT.glob("*.pt")):
            files.download(str(f))
    except Exception as e:
        print("Unduh otomatis tidak tersedia (", e, "). Ambil dari Google Drive atau /content/out/.")

tersimpan: /content/out/ppe-v2-ft1.pt | kelas: ['helmet', 'gloves', 'vest', 'boots', 'goggles', 'Person', 'no_helmet', 'no_goggle', 'no_gloves', 'no_boots', 'forklift', 'no_vest', 'mask', 'no_mask']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Setelah selesai

1. Salin `ppe-v2-ft1.pt` ke laptop sebagai `runs/ppe-v2-ft1/weights/best.pt`.
2. Cocokkan daftar kelas yang dicetak sel 11 dengan `ALIASES` di `safework/labels.py`. Kelas baru: `mask` dan `no_mask`.
3. Jalankan `python -m safework --weights runs\ppe-v2-ft1\weights\best.pt --debug` dan uji dengan alat peraga di jarak pos.
4. Data publik tidak sama dengan kamera dan jarak pos kamu. Bila `mask` atau `no_mask` masih lemah, tambahkan 100 sampai 200 foto lokal (`scripts/capture_photos.py`) lalu latih lanjutan dengan `RUN_TAG` baru.